# 01 — Economic Data Preparation (Eurostat)

## Objective

Collect and prepare economic data at the NUTS2 level (2010–2023).

---

## Data

- Tourism (Eurostat — `tour_occ_nin2`)  
- GDP (Eurostat — `nama_10r_2gdp`)  
- Population (Eurostat — `demo_r_pjanaggr3`)  

---

## Method

- Filter relevant variables  
- Reshape datasets to long format  
- Harmonize identifiers:
  - NUTS2 regions  
  - Annual observations  

Data cleaning:
- Keep valid numeric values  
- Remove duplicates  
- Restrict to NUTS2 regions  

---

## Output

Prepared datasets:

- tourism.csv  
- gdp.csv  
- population.csv  

Saved in:

`data/raw/`

---

In [1]:
import pandas as pd
import numpy as np
import eurostat
import statsmodels.formula.api as smf

c:\Users\henri_ugzoq54\AppData\Local\Programs\Python\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# load
tourism_raw = eurostat.get_data_df("tour_occ_nin2")

# filter
tourism_raw = tourism_raw[
    (tourism_raw["unit"] == "NR") &
    (tourism_raw["nace_r2"] == "I551")
]

# reshape
tourism = tourism_raw.melt(
    id_vars=["geo\\TIME_PERIOD"],
    var_name="year",
    value_name="tourism_nights"
)

tourism = tourism.rename(columns={"geo\\TIME_PERIOD": "region"})

# clean
tourism["year"] = pd.to_numeric(tourism["year"], errors="coerce")
tourism["tourism_nights"] = pd.to_numeric(tourism["tourism_nights"], errors="coerce")

tourism = tourism.dropna()
tourism = tourism[tourism["region"].str.len() == 4]
tourism = tourism[tourism["year"] >= 2010]
tourism = tourism.drop_duplicates(["region", "year"])

print("Tourism:", tourism.shape)

Tourism: (4365, 3)


In [3]:
gdp_raw = eurostat.get_data_df("nama_10r_2gdp")

gdp = gdp_raw.copy()

if "na_item" in gdp.columns:
    gdp = gdp[gdp["na_item"] == "B1GQ"]

if "unit" in gdp.columns:
    gdp = gdp[gdp["unit"] == "MIO_EUR"]

year_cols = [c for c in gdp.columns if str(c).isdigit()]

gdp = gdp.melt(
    id_vars=["geo\\TIME_PERIOD"],
    value_vars=year_cols,
    var_name="year",
    value_name="gdp"
)

gdp = gdp.rename(columns={"geo\\TIME_PERIOD": "region"})

gdp["year"] = gdp["year"].astype(int)
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")

gdp = gdp[gdp["region"].str.len() == 4]
gdp = gdp.dropna(subset=["gdp"])

gdp = gdp.drop_duplicates(["region","year"])

print("GDP:", gdp.shape)

GDP: (7498, 3)


In [4]:
pop_raw = eurostat.get_data_df("demo_r_pjanaggr3")

pop = pop_raw.copy()

if "age" in pop.columns:
    pop = pop[pop["age"] == "TOTAL"]

if "sex" in pop.columns:
    pop = pop[pop["sex"] == "T"]

if "unit" in pop.columns:
    pop = pop[pop["unit"] == "NR"]

year_cols = [c for c in pop.columns if str(c).isdigit()]

pop = pop.melt(
    id_vars=["geo\\TIME_PERIOD"],
    value_vars=year_cols,
    var_name="year",
    value_name="population"
)

pop = pop.rename(columns={"geo\\TIME_PERIOD": "region"})

pop["year"] = pop["year"].astype(int)
pop["population"] = pd.to_numeric(pop["population"], errors="coerce")

pop = pop[pop["region"].str.len() == 4]
pop = pop.dropna(subset=["population"])

pop = pop.drop_duplicates(["region","year"])

print("POP:", pop.shape)

POP: (9509, 3)


In [5]:
tourism.to_csv("../data/raw/tourism.csv", index=False)
gdp.to_csv("../data/raw/gdp.csv", index=False)
pop.to_csv("../data/raw/population.csv", index=False)

In [6]:
print("Tourism:", tourism.shape)
print("GDP:", gdp.shape)
print("Population:", pop.shape)

Tourism: (4365, 3)
GDP: (7498, 3)
Population: (9509, 3)
